# SciNet - SciPy for .NET

Special functions, optimisation, integration and ODEs, interpolation, linear algebra, sparse
matrices, spatial algorithms, clustering, signal processing, image processing and statistics -
the `scipy.*` modules, on top of GraviNum.

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

Build first: `dotnet build src/SciNet -c Release`.

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/SciNet/bin/Release/net10.0/Gravicode.Science.SciNet.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviNum;
using Gravicode.Science.SciNet;
using Gravicode.Science.SciNet.Sparse;
using Gravicode.Science.SciNet.Spatial;

Console.WriteLine(SciNetInfo.Banner());

## Special functions (`scipy.special`)

Evaluated in the regions where naive formulas fail: `Erfc` far in the tail is computed directly
rather than as `1 - Erf`, and `LogNdtr` stays finite where `Ndtr` underflows.

In [ ]:
Console.WriteLine($"Gamma(0.5)   = {Special.Gamma(0.5):R}");
Console.WriteLine($"Jv(2.5, 10)  = {Special.Jv(2.5, 10):R}");
Console.WriteLine($"Erfc(6)      = {Special.Erfc(6):E6}");
Console.WriteLine($"LogNdtr(-40) = {Special.LogNdtr(-40):F6}");
var legendre = Special.RootsLegendre(5);
Console.WriteLine($"5-point Gauss-Legendre of x^8: {legendre.Integrate(x => Math.Pow(x, 8)):R} (exact {2.0 / 9:R})");
Console.WriteLine($"Planck constant: {Constants.Value("Planck constant")} {Constants.Unit("Planck constant")}");

In [ ]:
var besselPlot = new ScottPlot.Plot();
var besselX = NdArray.Linspace(0, 20, 400).ToArray();
foreach (var order in new[] { 0, 1, 2, 5 })
{
    var curve = besselPlot.Add.Scatter(besselX, besselX.Select(x => Special.Jv(order, x)).ToArray());
    curve.LegendText = $"J{order}(x)";
    curve.MarkerSize = 0;
}
besselPlot.Title("Bessel functions of the first kind");
besselPlot.ShowLegend();
besselPlot.GetPngHtml(800, 450)

## Optimisation (`scipy.optimize`)

Same algorithms as SciPy - on Rosenbrock, BFGS and L-BFGS-B take the same number of iterations
and function evaluations. The difference is that the objective is a .NET delegate, not a Python
call.

In [ ]:
double[] rosenStart = [1.3, 0.7, 0.8, 1.9, 1.2];
foreach (var method in new[] { MinimizeMethod.NelderMead, MinimizeMethod.Bfgs, MinimizeMethod.LBfgsB, MinimizeMethod.TrustNcg })
{
    var result = Optimize.Minimize(Optimize.Rosen, rosenStart, method,
        jac: method == MinimizeMethod.NelderMead ? null : Optimize.RosenDer,
        hessp: method == MinimizeMethod.TrustNcg ? Optimize.RosenHessProd : null,
        options: method == MinimizeMethod.NelderMead ? new NelderMeadOptions { Xatol = 1e-8 } : null);
    Console.WriteLine($"{method,-11} f = {result.Fun:E2}  nit {result.Nit,4}  nfev {result.Nfev,4}");
}

In [ ]:
var decayT = Enumerable.Range(0, 50).Select(i => i * 0.1).ToArray();
var decayY = decayT.Select((t, i) => 2.5 * Math.Exp(-1.3 * t) + 0.5 + 0.05 * Math.Sin(17.0 * i)).ToArray();
var decayFit = Optimize.CurveFit((t, p) => p[0] * Math.Exp(-p[1] * t) + p[2], decayT, decayY, p0: [1, 1, 1]);
Console.WriteLine($"a = {decayFit.Popt[0]:F4} +/- {decayFit.Perr[0]:F4}");
Console.WriteLine($"b = {decayFit.Popt[1]:F4} +/- {decayFit.Perr[1]:F4}");
Console.WriteLine($"c = {decayFit.Popt[2]:F4} +/- {decayFit.Perr[2]:F4}");

var fitPlot = new ScottPlot.Plot();
var observed = fitPlot.Add.Scatter(decayT, decayY);
observed.LineWidth = 0;
observed.LegendText = "data";
var fitted = fitPlot.Add.Scatter(decayT, decayT.Select(t => decayFit.Popt[0] * Math.Exp(-decayFit.Popt[1] * t) + decayFit.Popt[2]).ToArray());
fitted.MarkerSize = 0;
fitted.LegendText = "curve_fit";
fitPlot.ShowLegend();
fitPlot.GetPngHtml(800, 400)

In [ ]:
var rastrigin = Optimize.DifferentialEvolution(x => 10 * x.Length + x.Sum(v => v * v - 10 * Math.Cos(2 * Math.PI * v)),
    new Bounds([-5.12, -5.12, -5.12], [5.12, 5.12, 5.12]), seed: 7);
Console.WriteLine($"differential evolution, Rastrigin-3: f = {rastrigin.Fun:E2}");

var lpResult = Optimize.Linprog([-1, 4], aUb: NdArray.FromRows([[-3, 1], [1, 2]]), bUb: [6, 4]);
Console.WriteLine($"linprog: x = [{string.Join(", ", lpResult.X)}], objective {lpResult.Fun}");

var assignmentCosts = NdArray.FromRows([[4, 1, 3], [2, 0, 5], [3, 2, 2]]);
var (assignedRows, assignedCols) = Optimize.LinearSumAssignment(assignmentCosts);
Console.WriteLine($"assignment: {string.Join(", ", assignedRows.Zip(assignedCols, (r, c) => $"{r}->{c}"))}");

## Integration and ODEs (`scipy.integrate`)

`Quad` is a line-by-line QUADPACK port, so it returns SciPy's value, error estimate and evaluation
count. The ODE right-hand side is an allocation-free delegate.

In [ ]:
var logQuad = Integrate.Quad(x => Math.Log(x) / Math.Sqrt(x), 0, 1);
Console.WriteLine($"quad log(x)/sqrt(x) = {logQuad.Value:R}  abserr {logQuad.AbsErr:E2}  neval {logQuad.Neval}");

void LorenzRhs(double t, ReadOnlySpan<double> y, Span<double> dy)
{
    dy[0] = 10 * (y[1] - y[0]);
    dy[1] = y[0] * (28 - y[2]) - y[1];
    dy[2] = y[0] * y[1] - 8.0 / 3.0 * y[2];
}

var lorenzRun = Integrate.SolveIvp(LorenzRhs, 0, 40, [1.0, 1.0, 1.0], OdeMethod.Dop853, rtol: 1e-9, atol: 1e-12, denseOutput: true);
Console.WriteLine($"{lorenzRun.T.Length} steps, nfev {lorenzRun.Nfev}");

var lorenzDense = lorenzRun.Sol!.Evaluate(Enumerable.Range(0, 8000).Select(i => i * 40.0 / 7999).ToArray());
var attractorPlot = new ScottPlot.Plot();
var attractor = attractorPlot.Add.Scatter(lorenzDense.Row(0).ToArray(), lorenzDense.Row(2).ToArray());
attractor.MarkerSize = 0;
attractor.LineWidth = 1;
attractorPlot.Title("Lorenz attractor");
attractorPlot.GetPngHtml(800, 600)

## Interpolation (`scipy.interpolate`)

In [ ]:
var splineKnots = Enumerable.Range(0, 8).Select(i => i * 1.0).ToArray();
var splineValues = new[] { 0.0, 0.8, 0.9, 0.1, -0.8, -1.0, -0.1, 0.6 };
var natural = new CubicSpline(splineKnots, splineValues, SplineBoundaryCondition.Natural);
var monotone = new PchipInterpolator(splineKnots, splineValues);
var akima = new Akima1DInterpolator(splineKnots, splineValues);
var fine = NdArray.Linspace(0, 7, 300).ToArray();

var interpPlot = new ScottPlot.Plot();
var knotMarks = interpPlot.Add.Scatter(splineKnots, splineValues);
knotMarks.LineWidth = 0;
knotMarks.MarkerSize = 9;
knotMarks.LegendText = "knots";
foreach (var (name, interpolant) in new (string, PiecewiseInterpolantBase)[] { ("natural cubic spline", natural), ("PCHIP", monotone), ("Akima", akima) })
{
    var line = interpPlot.Add.Scatter(fine, interpolant.Evaluate(fine));
    line.MarkerSize = 0;
    line.LegendText = name;
}
interpPlot.ShowLegend();
interpPlot.GetPngHtml(800, 450)

## Linear algebra (`scipy.linalg`)

In [ ]:
var symmetric = NdArray.FromRows([[4, 1, 0], [1, 3, 1], [0, 1, 2]]);
var squareRoot = Linalg.Sqrtm(symmetric);
Console.WriteLine($"sqrtm(A)^2 - A: {(LinAlg.Dot(squareRoot, squareRoot) - symmetric).ToArray().Max(Math.Abs):E2}");
var quarterTurn = Linalg.Expm(NdArray.FromRows([[0, -1], [1, 0]]) * (Math.PI / 2));
Console.WriteLine(quarterTurn.ToString(6));
var sylvester = Linalg.SolveSylvester(symmetric, symmetric, NdArray.Eye(3));
Console.WriteLine($"Sylvester residual: {(LinAlg.Dot(symmetric, sylvester) + LinAlg.Dot(sylvester, symmetric) - NdArray.Eye(3)).ToArray().Max(Math.Abs):E2}");

## Sparse matrices and graphs (`scipy.sparse`, `scipy.sparse.linalg`, `scipy.sparse.csgraph`)

In [ ]:
const int poissonSide = 100;
var tridiagonal = SparseMatrices.Diags([Enumerable.Repeat(-1.0, poissonSide - 1).ToArray(),
    Enumerable.Repeat(2.0, poissonSide).ToArray(), Enumerable.Repeat(-1.0, poissonSide - 1).ToArray()], [-1, 0, 1]);
var laplacian2d = SparseMatrices.Kronsum(tridiagonal, tridiagonal);
var cgSolution = SparseLinalg.Cg(laplacian2d, Enumerable.Repeat(1.0, poissonSide * poissonSide).ToArray(), rtol: 1e-10, maxiter: 5000);
Console.WriteLine($"CG: converged {cgSolution.Converged} in {cgSolution.Iterations} iterations");

var potential = new double[poissonSide, poissonSide];
for (var i = 0; i < poissonSide; i++)
    for (var j = 0; j < poissonSide; j++)
        potential[i, j] = cgSolution.X[i * poissonSide + j];
var heatPlot = new ScottPlot.Plot();
heatPlot.Add.Heatmap(potential);
heatPlot.Title("Solution of the 2-D Poisson problem");
heatPlot.GetPngHtml(600, 550)

## Spatial algorithms and clustering (`scipy.spatial`, `scipy.cluster`)

In [ ]:
var cloudRng = new GraviRandom(3);
var scattered = cloudRng.Uniform(0, 1, 60, 2);
var mesh = new Delaunay(scattered);
var meshPlot = new ScottPlot.Plot();
for (var t = 0; t < mesh.Simplices.GetLength(0); t++)
{
    var corners = Enumerable.Range(0, 4).Select(k => mesh.Simplices[t, k % 3]).ToArray();
    var edge = meshPlot.Add.Scatter(corners.Select(c => scattered[c, 0]).ToArray(), corners.Select(c => scattered[c, 1]).ToArray());
    edge.MarkerSize = 0;
    edge.Color = ScottPlot.Colors.SteelBlue;
}
meshPlot.Title($"Delaunay triangulation: {mesh.Simplices.GetLength(0)} triangles");
meshPlot.GetPngHtml(600, 600)

In [ ]:
var irisRows = File.ReadAllLines("../datasets/iris.csv").Skip(1).Where(l => l.Length > 0)
    .Select(l => l.Split(',').Take(4).Select(v => double.Parse(v, System.Globalization.CultureInfo.InvariantCulture)).ToArray()).ToArray();
var irisData = NdArray.FromRows(irisRows);
var wardTree = Cluster.Linkage(irisData, LinkageMethod.Ward);
var irisClusters = Cluster.Fcluster(wardTree, 3, FclusterCriterion.Maxclust);
Console.WriteLine($"cluster sizes: {string.Join(", ", irisClusters.GroupBy(c => c).OrderBy(g => g.Key).Select(g => g.Count()))}");
var neighbours = new KDTree(irisData).Query(irisRows[0], k: 4);
Console.WriteLine($"nearest neighbours of sample 0: {string.Join(", ", neighbours.Indices.Skip(1))}");

## Signal processing and FFT (`scipy.signal`, `scipy.fft`)

In [ ]:
const double sampleRate = 1000.0;
var times = Enumerable.Range(0, 2000).Select(i => i / sampleRate).ToArray();
var hummed = times.Select((t, i) => Math.Sin(2 * Math.PI * 5 * t) + 0.4 * Math.Sin(2 * Math.PI * 120 * t)).ToArray();
var lowpass = Signal.Butter(4, 20.0, FilterType.Lowpass, fs: sampleRate);
var cleaned = Signal.Filtfilt(lowpass.B, lowpass.A, hummed);
var (_, cutoffResponse) = Signal.Freqz(lowpass.B, lowpass.A, [20.0], fs: sampleRate);
Console.WriteLine($"gain at the 20 Hz cutoff: {20 * Math.Log10(cutoffResponse[0].Magnitude):F2} dB (Butterworth: -3.01 dB)");

var signalPlot = new ScottPlot.Plot();
var hum = signalPlot.Add.Scatter(times, hummed);
hum.MarkerSize = 0;
hum.LegendText = "5 Hz + 120 Hz";
var zeroPhase = signalPlot.Add.Scatter(times, cleaned);
zeroPhase.MarkerSize = 0;
zeroPhase.LineWidth = 3;
zeroPhase.LegendText = "filtfilt";
signalPlot.ShowLegend();
signalPlot.GetPngHtml(900, 400)

In [ ]:
var welchNoise = new GraviRandom(11);
var longSignal = Enumerable.Range(0, 200_000).Select(i =>
    Math.Sin(2 * Math.PI * 50 * i / sampleRate) + 0.5 * Math.Sin(2 * Math.PI * 120 * i / sampleRate)
    + 0.2 * Math.Sin(2 * Math.PI * 310 * i / sampleRate) + welchNoise.Normal(0, 1.0)).ToArray();
var (psdFreqs, psdValues) = Signal.Welch(longSignal, fs: sampleRate, nperseg: 2048);
var psdDb = psdValues.Select(v => 10 * Math.Log10(v)).ToArray();
var spectralPeaks = Signal.FindPeaks(psdDb, prominence: 10.0);
Console.WriteLine($"tones found at: {string.Join(", ", spectralPeaks.Peaks.Select(k => $"{psdFreqs[k]:F1} Hz"))}");

var welchPlot = new ScottPlot.Plot();
var psdLine = welchPlot.Add.Scatter(psdFreqs, psdDb);
psdLine.MarkerSize = 0;
var peakMarks = welchPlot.Add.Scatter(spectralPeaks.Peaks.Select(k => psdFreqs[k]).ToArray(), spectralPeaks.Peaks.Select(k => psdDb[k]).ToArray());
peakMarks.LineWidth = 0;
peakMarks.MarkerSize = 10;
welchPlot.XLabel("frequency (Hz)");
welchPlot.YLabel("PSD (dB/Hz)");
welchPlot.GetPngHtml(900, 400)

## Statistics (`scipy.stats`)

In [ ]:
var iq = Stats.Norm(loc: 100, scale: 15);
Console.WriteLine($"P(IQ > 130) = {iq.Sf(130):F5},  99th percentile = {iq.Ppf(0.99):F2}");
var gammaDraws = Stats.Gamma(a: 2.5, scale: 1.5).Rvs(20_000, seed: 3);
var gammaFit = (GammaDistribution)Stats.Gamma(a: 1).FitTo(gammaDraws, floc: 0);
Console.WriteLine($"fitted gamma: a = {gammaFit.A:F3}, scale = {gammaFit.Scale:F3}");

var bins = 60;
var (lowEdge, highEdge) = (0.0, 15.0);
var counts = new double[bins];
foreach (var v in gammaDraws) if (v < highEdge) counts[(int)(v / (highEdge - lowEdge) * bins)]++;
var centres = Enumerable.Range(0, bins).Select(i => (i + 0.5) * (highEdge - lowEdge) / bins).ToArray();
var binWidth = (highEdge - lowEdge) / bins;
var distPlot = new ScottPlot.Plot();
var histogram = distPlot.Add.Bars(centres, counts.Select(c => c / (gammaDraws.Length * binWidth)).ToArray());
histogram.LegendText = "draws";
var pdfCurve = distPlot.Add.Scatter(centres, centres.Select(x => gammaFit.Pdf(x)).ToArray());
pdfCurve.MarkerSize = 0;
pdfCurve.LineWidth = 3;
pdfCurve.LegendText = "fitted pdf";
distPlot.ShowLegend();
distPlot.GetPngHtml(800, 400)

In [ ]:
var setosa = irisRows.Take(50).Select(r => r[0]).ToArray();
var versicolor = irisRows.Skip(50).Take(50).Select(r => r[0]).ToArray();
var tTest = Stats.TtestInd(setosa, versicolor, equalVar: false);
Console.WriteLine($"Welch t = {tTest.Statistic:F3}, p = {tTest.Pvalue:E3}");
Console.WriteLine($"Shapiro-Wilk p (setosa) = {Stats.Shapiro(setosa).Pvalue:F4}");
var petalFit = Stats.Linregress(irisRows.Select(r => r[2]).ToArray(), irisRows.Select(r => r[3]).ToArray());
Console.WriteLine($"petal width ~ {petalFit.Slope:F4} x length, r = {petalFit.Rvalue:F4}, p = {petalFit.Pvalue:E2}");
var sobolPoints = new SobolEngine(2, scramble: false).RandomBase2(8);
Console.WriteLine($"Sobol discrepancy {Qmc.Discrepancy(sobolPoints):E3} vs random {Qmc.Discrepancy(new GraviRandom(5).Random(256, 2)):E3}");

## Image processing (`scipy.ndimage`)

In [ ]:
var shapes = NdArray.Zeros(64, 64);
for (var i = 0; i < 64; i++)
    for (var j = 0; j < 64; j++)
        if ((i - 20) * (i - 20) + (j - 20) * (j - 20) < 100 || (i > 38 && i < 56 && j > 34 && j < 58)) shapes[i, j] = 1;
var (objectLabels, objectCount) = NdImage.Label(shapes);
var distanceMap = NdImage.DistanceTransformEdt(shapes);
Console.WriteLine($"{objectCount} objects; deepest pixel {Statistics.Max(distanceMap):F3} from the background");

var distanceGrid = distanceMap.To2DArray();
var edtPlot = new ScottPlot.Plot();
edtPlot.Add.Heatmap(distanceGrid);
edtPlot.Title("Euclidean distance transform");
edtPlot.GetPngHtml(550, 550)